# Feature Scaling

Feature scaling is a method used to normalize the range of independent variables or features of data. It is a crucial preprocessing step for many machine learning algorithms.

## Why does Feature Scaling matter?

1. **Distance-Based Algorithms**: Algorithms like K-Nearest Neighbors (KNN), Support Vector Machines (SVM), and K-Means clustering use distance metrics (like Euclidean distance). If one feature has a range of 0-1000 and another has 0-1, the distance will be entirely dominated by the first feature.
2. **Gradient Descent**: Algorithms that use Gradient Descent for optimization (like Linear Regression, Logistic Regression, Neural Networks) converge much faster when features are on a similar scale. Unscaled data causes the cost function to look like an elongated bowl, leading to a slow, zig-zagging descent.
3. **Regularization**: Penalties in L1/L2 regularization assume all features are centered around 0 and have the same variance. Unscaled features will be penalized unfairly.

## 1. Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

## 2. Generating Sample Data

We will create a dataset with features on vastly different scales and with outliers.

In [ ]:
np.random.seed(42)
n_samples = 1000

# Feature 1: Normal distribution, mean 50, std 10
f1 = np.random.normal(50, 10, n_samples)

# Feature 2: Large scale, mean 10000, std 2000
f2 = np.random.normal(10000, 2000, n_samples)

# Feature 3: Small scale with outliers
f3 = np.random.normal(1, 0.5, n_samples)
f3[0:50] = f3[0:50] * 10 # Adding outliers

df = pd.DataFrame({'Age': f1, 'Income': f2, 'Score': f3})
print(df.describe())

## 3. Visualizing Original Data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(df['Age'], ax=axes[0], kde=True)
axes[0].set_title('Age (Normal Scale)')
sns.histplot(df['Income'], ax=axes[1], kde=True)
axes[1].set_title('Income (Large Scale)')
sns.histplot(df['Score'], ax=axes[2], kde=True)
axes[2].set_title('Score (Small Scale + Outliers)')
plt.tight_layout()
plt.show()

## 4. Standardization (StandardScaler)

Transforms features to have a mean of 0 and standard deviation of 1. Formula: `z = (x - u) / s`
It assumes your data is normally distributed within each feature and will scale them such that the distribution is centered around 0, with a standard deviation of 1.

In [ ]:
scaler = StandardScaler()
df_standard = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

print('Mean:\n', np.round(df_standard.mean(), 4))
print('Std:\n', df_standard.std())

## 5. Normalization (MinMaxScaler)

Scales features to a fixed range, typically between 0 and 1. Formula: `X_std = (X - X.min) / (X.max - X.min)`
Very useful when you need bounded values (e.g., for Image processing, where pixels are 0-255). Highly sensitive to outliers because they dictate the min and max.

In [ ]:
minmax = MinMaxScaler()
df_minmax = pd.DataFrame(minmax.fit_transform(df), columns=df.columns)

print('Min:\n', df_minmax.min())
print('Max:\n', df_minmax.max())

## 6. RobustScaler

Scales features using statistics that are robust to outliers. It removes the median and scales the data according to the Interquartile Range (IQR).
Formula: `(X - median) / IQR`. Excellent choice when your data contains significant outliers.

In [ ]:
robust = RobustScaler()
df_robust = pd.DataFrame(robust.fit_transform(df), columns=df.columns)

print(df_robust.describe())

## 7. Comparison of Scalers on Outliers

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

sns.boxplot(y=df['Score'], ax=axes[0])
axes[0].set_title('Original Score')

sns.boxplot(y=df_standard['Score'], ax=axes[1])
axes[1].set_title('StandardScaler')

sns.boxplot(y=df_minmax['Score'], ax=axes[2])
axes[2].set_title('MinMaxScaler')

sns.boxplot(y=df_robust['Score'], ax=axes[3])
axes[3].set_title('RobustScaler')

plt.tight_layout()
plt.show()

## 8. Practical Example: Impact on KNN Performance

Let's see how much scaling actually impacts a distance-based algorithm like K-Nearest Neighbors.

In [ ]:
# Create a synthetic classification target based on Age and Income
df['Target'] = ((df['Age'] > 50) & (df['Income'] > 10000)).astype(int)

X = df[['Age', 'Income', 'Score']]
y = df['Target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. KNN WITHOUT Scaling
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_unscaled.fit(X_train, y_train)
pred_unscaled = knn_unscaled.predict(X_test)
acc_unscaled = accuracy_score(y_test, pred_unscaled)

# 2. KNN WITH StandardScaler
scaler_demo = StandardScaler()
X_train_scaled = scaler_demo.fit_transform(X_train)
X_test_scaled = scaler_demo.transform(X_test) # ALWAYS transform test set, NEVER fit on it

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
pred_scaled = knn_scaled.predict(X_test_scaled)
acc_scaled = accuracy_score(y_test, pred_scaled)

print(f'Accuracy WITHOUT Scaling: {acc_unscaled:.4f}')
print(f'Accuracy WITH StandardScaler: {acc_scaled:.4f}')
print('Notice how scaling massively improves the performance of KNN!')

## Summary

- **StandardScaler**: Good general purpose default. Assumes data is normally distributed.
- **MinMaxScaler**: Good for image data or when you need a bounded range. Highly sensitive to outliers.
- **RobustScaler**: The best choice when your data contains significant outliers.
- **Always Fit on Train, Transform on Test**: Never fit your scaler on the test data, as this causes data leakage!